In [24]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from itertools import combinations
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import TensorDataset, DataLoader
from joblib import Parallel, delayed

In [25]:
# ─── Utilities ────────────────────────────────────────────────────────────────
lam, p, niter = 1e4, 0.01, 10
def baseline_als(y):
    L = len(y)
    D = np.diff(np.eye(L), 2)
    D = lam * D.dot(D.T)
    w = np.ones(L)
    for _ in range(niter):
        b = np.linalg.solve(np.diag(w) + D, w * y)
        w = p * (y > b) + (1 - p) * (y < b)
    return b

def preprocess(arr):
    """
    (Batch) baseline-correct → first derivative → L2-normalize
    """
    out = np.zeros_like(arr)
    for i, s in enumerate(arr):
        b = baseline_als(s)
        c = s - b
        d = np.gradient(c)             # <-- FIRST DERIVATIVE
        norm = np.linalg.norm(d)
        out[i] = d / norm if norm > 0 else d
    return out

def floatify_cols(df):
    new = []
    for c in df.columns:
        if c in ('Label', 'Label 1', 'Label 2'):
            new.append(c)
        else:
            new.append(float(c))
    df.columns = new

In [26]:
# ─── 1) Load reference_v2 and preprocess ──────────────────────────────────────
ref_df = pd.read_csv('reference_v2.csv')

floatify_cols(ref_df)
wav_cols = [c for c in ref_df.columns if c != 'Label']
ref_specs  = ref_df[wav_cols].values       # (n_ref_samples, n_waves)
ref_labels = ref_df['Label'].values        # (n_ref_samples,)

In [27]:
# Unique chemical classes
classes    = sorted(np.unique(ref_labels))
C = len(classes)
class_to_i = {c:i for i,c in enumerate(classes)}

# ─── 2) Generate synthetic mixtures ────────────────────────────────────────────
ratios = np.arange(0.05, 1.0, 0.05)
noise_level = 0.01
n_per_ratio = 10  # number of random spectra per pair/ratio

synth_specs = []
synth_labels = []
for (i, ci), (j, cj) in combinations(enumerate(classes), 2):
    # indices of pure spectra for each class
    idx_i = np.where(ref_labels == ci)[0]
    idx_j = np.where(ref_labels == cj)[0]
    for r in ratios:
        for _ in range(n_per_ratio):
            spec_i = ref_specs[np.random.choice(idx_i)]
            spec_j = ref_specs[np.random.choice(idx_j)]
            mix = r * spec_i + (1-r) * spec_j
            mix += np.random.normal(scale=noise_level, size=mix.shape)
            synth_specs.append(mix)
            synth_labels.append((ci, cj))
synth_specs = np.array(synth_specs)        # (n_synth, n_waves)
print("Synthetic raw spectra:", synth_specs.shape)

Synthetic raw spectra: (12540, 1024)


In [28]:
# ─── 3) Derivative-based preprocessing for synthetic ──────────────────────────
def preprocess_single(spectrum):
    """
    Baseline-correct → first derivative → L2-normalize (single spectrum).
    """
    b = baseline_als(spectrum)
    c = spectrum - b
    d = np.gradient(c)                 # <-- FIRST DERIVATIVE
    norm = np.linalg.norm(d)
    out = d / norm if norm > 0 else d
    return out

synth_proc = np.vstack(
    Parallel(n_jobs=-1, verbose=10)(
        delayed(preprocess_single)(spec) 
        for spec in synth_specs
    )
)
print("Parallel preprocess (derivative) done:", synth_proc.shape)

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  21 tasks      | elapsed:    1.2s
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    1.9s
[Parallel(n_jobs=-1)]: Done  49 tasks      | elapsed:    2.2s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:    2.5s
[Parallel(n_jobs=-1)]: Done  81 tasks      | elapsed:    3.2s
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed:    3.7s
[Parallel(n_jobs=-1)]: Done 117 tasks      | elapsed:    4.3s
[Parallel(n_jobs=-1)]: Done 136 tasks      | elapsed:    4.8s
[Parallel(n_jobs=-1)]: Done 157 tasks      | elapsed:    5.4s
[Parallel(n_jobs=-1)]: Done 178 tasks      | elapsed:    6.2s
[Parallel(n_jobs=-1)]: Done 201 tasks      | elapsed:    7.0s
[Parallel(n_jobs=-1)]: Done 224 tasks      | elapsed:    7.6s
[Parallel(n_jobs=-1)]: Done 249 tasks      | elapsed:    8.4s
[Parallel(n_jobs=-1)]: Done 274 tasks      | elapsed:  

Parallel preprocess (derivative) done: (12540, 1024)


[Parallel(n_jobs=-1)]: Done 12540 out of 12540 | elapsed:  6.5min finished


In [29]:
# ─── 4) Embed synthetic mixtures via Siamese ──────────────────────────────────
# load siamese
class SiameseNet(nn.Module):
    def __init__(self, input_len, embed_dim=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1,16,7,padding=3), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16,32,5,padding=2), nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Flatten(),
            nn.Linear((input_len//4)*32, embed_dim), nn.ReLU()
        )
    def forward(self,x):
        z = self.encoder(x)
        return F.normalize(z, dim=1)

siamese = SiameseNet(input_len=ref_specs.shape[1], embed_dim=64)
siamese.load_state_dict(torch.load('siamese_mixture_deriv.pth', map_location='cpu'))
siamese.eval()

with torch.no_grad():
    tensor = torch.tensor(synth_proc, dtype=torch.float32).unsqueeze(1)
    syn_embeds = siamese(tensor).cpu().numpy()  # (n_synth, 64)
print("Synthetic embeddings:", syn_embeds.shape)

Synthetic embeddings: (12540, 64)


In [30]:
# ─── 5) Build X_synth, Y_synth ─────────────────────────────────────────────────
N = len(syn_embeds)
X_synth = syn_embeds
Y_synth = np.zeros((N, C), dtype=int)
for k, (ci, cj) in enumerate(synth_labels):
    Y_synth[k, class_to_i[ci]] = 1
    Y_synth[k, class_to_i[cj]] = 1

In [31]:
# ─── 6) Split synthetic into train/val/test (80/10/10) ─────────────────────────
X_tmp, X_test_s, Y_tmp, Y_test_s = train_test_split(X_synth, Y_synth, test_size=0.10, random_state=0)
X_train_s, X_val_s, Y_train_s, Y_val_s = train_test_split(X_tmp, Y_tmp, test_size=0.1111, random_state=0)
print("Synthetic train/val/test:", len(X_train_s), len(X_val_s), len(X_test_s))

Synthetic train/val/test: 10032 1254 1254


In [32]:
# ─── 7) DataLoaders for synthetic ─────────────────────────────────────────────
batch_size = 64
train_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_train_s, dtype=torch.float32),
        torch.tensor(Y_train_s, dtype=torch.float32)   # ← make this float
    ),
    batch_size=batch_size,
    shuffle=True
)
val_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_val_s, dtype=torch.float32),
        torch.tensor(Y_val_s, dtype=torch.float32)     # ← and this
    ),
    batch_size=batch_size
)
test_loader = DataLoader(
    TensorDataset(
        torch.tensor(X_test_s, dtype=torch.float32),
        torch.tensor(Y_test_s, dtype=torch.float32)    # ← and this
    ),
    batch_size=batch_size
)


In [33]:
# 1) Define the PresenceNet WITHOUT final Sigmoid
class PresenceNetLogits(nn.Module):
    def __init__(self, D, C):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(D, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, C)   # raw logits
        )
    def forward(self, x):
        return self.net(x)

In [34]:
# 2) Instantiate model
D = X_train_s.shape[1]  # embedding dimension
C = len(classes)
model_boost = PresenceNetLogits(D, C)



In [36]:
# 3) Build pos_weight to up-weight specific classes

pos = Y_train_s.sum(axis=0)
neg = len(Y_train_s) - pos
pos_weight = torch.tensor((neg/pos).clip(min=1.0), dtype=torch.float32)


print("pos_weight:", pos_weight)

pos_weight: tensor([4.9714, 5.0726, 5.0216, 4.9928, 5.0800, 4.9326, 5.1096, 5.0000, 4.9361,
        4.9750, 5.0144, 4.9012])


In [37]:
# 4) Use BCEWithLogitsLoss with pos_weight
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model_boost.parameters(), lr=1e-3)

# 5) Training loop skeleton
num_epochs = 200
for epoch in range(1, num_epochs+1):
    model_boost.train()
    train_loss = 0.0
    for xb, yb in train_loader:
        logits = model_boost(xb)
        loss = criterion(logits, yb)  # yb must be FloatTensor
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * xb.size(0)
    train_loss /= len(train_loader.dataset)

    # Validation...
    model_boost.eval()
    val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            logits = model_boost(xb)
            val_loss += criterion(logits, yb).item() * xb.size(0)
    val_loss /= len(val_loader.dataset)

    print(f"Epoch {epoch}/{num_epochs} — "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

Epoch 1/200 — Train Loss: 0.9186 | Val Loss: 0.6559
Epoch 2/200 — Train Loss: 0.5754 | Val Loss: 0.5229
Epoch 3/200 — Train Loss: 0.4982 | Val Loss: 0.4800
Epoch 4/200 — Train Loss: 0.4648 | Val Loss: 0.4531
Epoch 5/200 — Train Loss: 0.4407 | Val Loss: 0.4265
Epoch 6/200 — Train Loss: 0.4194 | Val Loss: 0.4087
Epoch 7/200 — Train Loss: 0.4008 | Val Loss: 0.3887
Epoch 8/200 — Train Loss: 0.3850 | Val Loss: 0.3731
Epoch 9/200 — Train Loss: 0.3720 | Val Loss: 0.3660
Epoch 10/200 — Train Loss: 0.3603 | Val Loss: 0.3539
Epoch 11/200 — Train Loss: 0.3510 | Val Loss: 0.3500
Epoch 12/200 — Train Loss: 0.3420 | Val Loss: 0.3365
Epoch 13/200 — Train Loss: 0.3351 | Val Loss: 0.3291
Epoch 14/200 — Train Loss: 0.3278 | Val Loss: 0.3221
Epoch 15/200 — Train Loss: 0.3224 | Val Loss: 0.3136
Epoch 16/200 — Train Loss: 0.3150 | Val Loss: 0.3082
Epoch 17/200 — Train Loss: 0.3094 | Val Loss: 0.3003
Epoch 18/200 — Train Loss: 0.3045 | Val Loss: 0.2962
Epoch 19/200 — Train Loss: 0.2995 | Val Loss: 0.2906
Ep

In [38]:
# --- Load mixtures and convert columns ---
mix_df = pd.read_csv('mixtures_dataset.csv')

# Re-use your floatify_cols helper:
def floatify_cols(df):
    new_cols = []
    for c in df.columns:
        if c in ('Label 1', 'Label 2'):
            new_cols.append(c)
        else:
            new_cols.append(float(c))  # convert wavenumber strings → floats
    df.columns = new_cols

floatify_cols(mix_df)

# --- Now select wavenumber columns (they are all numeric) ---
wav_cols = [c for c in mix_df.columns if c not in ('Label 1', 'Label 2')]

# --- Extract spectra as a pure float array ---
mix_specs = mix_df[wav_cols].values.astype(float)  # ensure float64 dtype

# --- Parallel preprocessing now works because mix_specs is numeric ---
mix_proc = np.vstack(
    Parallel(n_jobs=-1, verbose=5)(
        delayed(preprocess_single)(spec) for spec in mix_specs
    )
)
print("Mixtures preprocessed:", mix_proc.shape)
pairs     = list(zip(mix_df['Label 1'], mix_df['Label 2']))

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 32 concurrent workers.
[Parallel(n_jobs=-1)]: Done   8 tasks      | elapsed:    1.0s
[Parallel(n_jobs=-1)]: Done  98 tasks      | elapsed:    3.8s
[Parallel(n_jobs=-1)]: Done 224 tasks      | elapsed:    7.6s
[Parallel(n_jobs=-1)]: Done 386 tasks      | elapsed:   12.6s


Mixtures preprocessed: (580, 1024)


[Parallel(n_jobs=-1)]: Done 580 out of 580 | elapsed:   18.0s finished


In [39]:
with torch.no_grad():
    mix_embeds = siamese(torch.tensor(mix_proc, dtype=torch.float32).unsqueeze(1)).cpu().numpy()

# build real multi-hot
N_real = len(mix_df)
Y_real = np.zeros((N_real, C), dtype=int)
for i, (l1, l2) in enumerate(pairs):
    Y_real[i, class_to_i[l1]] = 1
    Y_real[i, class_to_i[l2]] = 1

# predict real
model_boost.eval()
preds = model_boost(torch.tensor(mix_embeds, dtype=torch.float32)).detach().numpy()
Y_pred_real = (preds>0.5).astype(int)

from sklearn.metrics import classification_report

# 1) Compute support for each class
supports = Y_real.sum(axis=0)   # length C array of counts

# 2) Select only the classes with support > 0
valid_idx = [i for i, s in enumerate(supports) if s > 0]
valid_labels = [classes[i] for i in valid_idx]

# 3) Filter y_true and y_pred to these columns
y_true_filt = Y_real[:, valid_idx]
y_pred_filt = Y_pred_real[:, valid_idx]

# 4) Print report on the filtered set
print("\nReal Mixtures Validation Report (labels with support > 0):")
print(classification_report(
    y_true_filt,
    y_pred_filt,
    target_names=valid_labels,
    zero_division=0
))


Real Mixtures Validation Report (labels with support > 0):
                       precision    recall  f1-score   support

      1-dodecanethiol       0.92      0.58      0.71       243
 6-mercapto-1-hexanol       0.99      0.67      0.80       108
              benzene       1.00      1.00      1.00       193
         benzenethiol       0.48      1.00      0.65        72
                 etoh       0.96      1.00      0.98       121
                 meoh       1.00      0.96      0.98       243
n,n-dimethylformamide       1.00      1.00      1.00        72
             pyridine       1.00      1.00      1.00       108

            micro avg       0.91      0.87      0.89      1160
            macro avg       0.92      0.90      0.89      1160
         weighted avg       0.94      0.87      0.89      1160
          samples avg       0.92      0.87      0.88      1160



In [40]:
# ─── 9) Evaluate synthetic test set ────────────────────────────────────────────
model_boost.eval()
yp, yt = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        yp.append(model_boost(xb).numpy())
        yt.append(yb.numpy())
y_pred = (np.vstack(yp)>0.5).astype(int)
y_true = np.vstack(yt)
print("\nSynthetic Test Classification Report:")
print(classification_report(y_true, y_pred, target_names=classes))


Synthetic Test Classification Report:
                              precision    recall  f1-score   support

           1,9-nonanedithiol       0.87      0.85      0.86       196
             1-dodecanethiol       0.55      0.91      0.69       225
             1-undecanethiol       0.50      0.92      0.65       208
        6-mercapto-1-hexanol       0.75      0.94      0.84       204
                     benzene       1.00      1.00      1.00       235
                benzenethiol       1.00      0.99      1.00       195
                        dmmp       1.00      0.99      0.99       220
                        etoh       0.97      0.94      0.95       217
                        meoh       0.94      0.93      0.93       184
       n,n-dimethylformamide       1.00      0.98      0.99       212
                    pyridine       1.00      0.98      0.99       223
tris(2-ethylhexyl) phosphate       0.99      0.95      0.97       189

                   micro avg       0.84      0.95